# 08 - Generation-response evaluation analysis

We analyse the 140 blinded judgments against the frozen 20-question, seven-condition design. The judgments were validated and imported before this stage. Our scores are final for analysis, while the professor's current feedback remains preliminary; later comments on the overall system can inform the thesis discussion without changing these scores. We preserve the exact judgment fingerprint and the separate private status records.

We only display and export aggregate counts, accuracy, secondary-score distributions, paired comparisons and descriptive strata. Candidate text, reference solutions, judgment notes, question identifiers and answer-level scores remain private. Notebook 07 records reference validation; the private grouped notebook holds the original scoring record.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs/generation-response-evaluation-config.json").is_file():
    raise FileNotFoundError("The thesis repository root was not found.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from geotech_rag.response_analysis import (
    compute_aggregates, load_frozen_analysis, output_payload, save_public_aggregates,
)

RUN_WRITE_PUBLIC_RESULTS = False
print("Repository:", PROJECT_ROOT)
print("Write public aggregate results:", RUN_WRITE_PUBLIC_RESULTS)

## 1. Recheck the frozen lineage

We verify the reference, question, generated response, completed review notebook, imported judgments and both private decision records against their saved fingerprints. We match all 140 blinded IDs to their frozen question-condition pairs only in memory. The score file contains neither condition identities nor response text. These checks stop analysis if one input has changed or a question-condition pair is missing.

The earlier private status record reports preliminary professor feedback. A later private decision records our instruction that the 140 scores are final for analysis. Neither record replaces the expert judgment file.

In [ ]:
config, questions, references, paired_rows, fingerprints = load_frozen_analysis(PROJECT_ROOT)
print("Verified frozen questions:", len(questions))
print("Verified paired judgments:", len(paired_rows))
print("Judgment SHA-256:", fingerprints["judgments_sha256"])
print("Score-set decision SHA-256:", fingerprints["final_score_set_decision_sha256"])
print("Question and answer text displayed: False")

## 2. Report answer correctness and completion

We use binary correctness as the primary outcome, with all 20 prevalidated questions in every condition. Accuracy is the number correct divided by 20. We report Wilson 95% confidence intervals and visible-response completion separately. The nine frozen responses without visible text stay in the accuracy denominator and score incorrect. We do not infer within-condition sampling variance from a single generated answer per question.

In [ ]:
aggregate = compute_aggregates(questions, references, paired_rows, fingerprints)
display(pd.DataFrame(aggregate["condition_summary"]))

## 3. Compare matched conditions

Every condition answered the same 20 questions. We compare P0 with P1 using exact McNemar; for P1/P2/P3 and P1/M1/M2/M3 we use Cochran's Q, then exact pairwise McNemar comparisons. Holm adjustment applies within each planned family. Tests with no discordance have p = 1. These are paired analyses; an unpaired chi-square calculation cannot replace them.

The original paper's GPT-4/Llama-3 pair is absent from our seven frozen conditions. We therefore do not report its unpaired chi-square comparison as though it were reproduced.

In [ ]:
display(pd.DataFrame.from_dict(aggregate["binary_omnibus"], orient="index").reset_index(names="family"))
display(pd.DataFrame(aggregate["binary_pairwise"]))

## 4. Describe secondary judgments and errors

We summarise formula integration, calculation correctness, unit correctness, explanation clarity and task adaptability on their frozen 0-2 scales. The first three use only questions marked applicable in the approved reference; the same eligible questions enter every compared condition. For two conditions we use paired Wilcoxon signed-rank tests, and for three or four conditions we use Friedman tests followed by paired Wilcoxon comparisons with Holm adjustment. Tied scores and pairs without a difference remain accounted for.

We report one primary error category per incorrect response, including the documented `no_visible_answer` extension. Problem-type and cognitive-level summaries are descriptive: ten reference records lack these classifications and are labelled `not_recorded_in_reference`, rather than being assigned a category from generated answers. Recorded categories with fewer than three questions are pooled before we publish results. All recorded problem types are pooled here; small strata are interpreted descriptively. We also count unsupported claims per condition.

In [ ]:
display(pd.DataFrame(aggregate["secondary_summary"]))
display(pd.DataFrame(aggregate["error_counts"]))
for family, comparisons in aggregate["ordinal_comparisons"].items():
    print("Ordinal comparison family:", family)
    for field, result in comparisons.items():
        print(field, "common applicable questions:", result["common_applicable_questions"],
              "Friedman:", result["friedman"])
        display(pd.DataFrame(result["paired_wilcoxon"]))
for field, records in aggregate["adaptability_strata"].items():
    print("Descriptive stratum:", field)
    display(pd.DataFrame(records))

## 5. Save only aggregate outputs

We first preview file paths, byte counts and fingerprints. The write flag remains disabled until the public-only results are checked. The five outputs contain the condition summary, applicable secondary scores, paired binary comparisons, error distributions and the full aggregate metric record. We create each output only once and refuse to replace an existing result silently. No private question, answer, note, response ID or per-question judgment is exported.

In [ ]:
payload = output_payload(PROJECT_ROOT, config, aggregate)
for path, content in payload.items():
    print(path.relative_to(PROJECT_ROOT), len(content), hashlib.sha256(content).hexdigest())
if RUN_WRITE_PUBLIC_RESULTS:
    save_public_aggregates(payload)
    print("Five aggregate outputs saved without overwrite.")
else:
    print("Public write disabled; preview only.")

## 6. Interpretation boundaries

We compare conditions under one frozen paired benchmark. We will discuss grouped presentation, one expert, the nine no-visible-answer responses and preliminary professor feedback when interpreting the results. We report the professor's later overall-system comment separately from answer scores. The baseline paper reports 82.5% in one setting, but the unpublished calculation behind that value cannot be reconstructed as binary correctness over exactly 20 questions; our score remains an explicit correct/20 measure. Any later correction to an individual score would require the documented private correction and analysis to be rerun before claims are final.